In [1]:
import pandas as pd
import numpy as np

# ── 1. LOAD ──────────────────────────────────────────────────────────────────
# Treat 'none', '?' and 'None' as NaN right at load time
df = pd.read_csv('/content/modified_adult.csv', na_values=['none', '?', 'None'])

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Missing values before cleaning:\n", df.isna().sum(), "\n")


# ── 2. DROP USELESS COLUMNS ──────────────────────────────────────────────────
# fnlwgt      : was entirely 'none' — zero information
# Person-Name : PII (first names), no predictive value
df = df.drop(columns=['fnlwgt', 'Person-Name'])


# ── 3. PARSE 'Details' COLUMN → race / sex / native-country ─────────────────
# Format: "<Race> <Sex> Person from <Country>"
# e.g.  "White Male Person from US"  /  "Asian-Pac-Islander Female Person from ?"
df['race']           = df['Details'].str.extract(r'^([\w\-]+)')
df['sex']            = df['Details'].str.extract(r'(Male|Female)')
df['native-country'] = df['Details'].str.extract(r'from\s+(\S+)$')

# '?' in native-country was already handled by na_values at load;
# re-apply to the freshly extracted column just in case
df['native-country'] = df['native-country'].replace('?', np.nan)

# Drop the raw text column now that we have extracted what we need
df = df.drop(columns=['Details'])


# ── 4. FIX DATA TYPES ────────────────────────────────────────────────────────
# age and educational-num were read as object because 'none' was mixed in.
# errors='coerce' turns any remaining non-numeric stragglers into NaN.
df['age']             = pd.to_numeric(df['age'],             errors='coerce')
df['educational-num'] = pd.to_numeric(df['educational-num'], errors='coerce')


# ── 5. STANDARDIZE INCONSISTENT CATEGORY LABELS ──────────────────────────────
# workclass: 'pvt' → 'Private', 'Local-gov' → 'Local-government'
df['workclass'] = df['workclass'].replace({
    'pvt'      : 'Private',
    'Local-gov': 'Local-government',
})


# ── 6. HANDLE MISSING VALUES ─────────────────────────────────────────────────
# Numeric columns  → fill with median  (robust to skew / outliers)
# Categorical cols → fill with mode    (most frequent value)

numeric_cols     = ['age', 'educational-num']
categorical_cols = ['workclass', 'occupation', 'marital-status', 'native-country']

for col in numeric_cols:
    median = df[col].median()
    df[col] = df[col].fillna(median)
    print(f"  Filled '{col}' NaNs with median = {median}")

for col in categorical_cols:
    mode = df[col].mode()[0]
    df[col] = df[col].fillna(mode)
    print(f"  Filled '{col}' NaNs with mode = '{mode}'")


# ── 7. CAP CAPITAL-GAIN SENTINEL VALUE ───────────────────────────────────────
# 99999 is a known capped/unknown sentinel in the Adult dataset.
# Replace with NaN then fill with median so it does not skew the distribution.
df['capital-gain'] = df['capital-gain'].replace(99999, np.nan)
df['capital-gain'] = df['capital-gain'].fillna(df['capital-gain'].median())


# ── 8. ENCODE TARGET LABEL AS BINARY INTEGER ─────────────────────────────────
# '<=50K' → 0,  '>50K' → 1
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})


# ── 9. RESET INDEX ───────────────────────────────────────────────────────────
df = df.reset_index(drop=True)


# ── 10. FINAL SANITY CHECK ───────────────────────────────────────────────────
print(f"\nCleaned: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Missing values after cleaning:\n", df.isna().sum())
print("\nData types:\n", df.dtypes)
print("\nSample rows:")
print(df.head(3).to_string())

# Save
df.to_csv('adult_clean.csv', index=False)
print("\nSaved → adult_clean.csv")

Loaded: 49,042 rows × 15 columns
Missing values before cleaning:
 ID                     0
age                 3933
workclass           2809
fnlwgt             49039
education              0
educational-num     3928
marital-status      3931
occupation          6529
relationship           0
capital-gain           0
capital-loss           0
hours-per-week         0
Person-Name            0
Details                0
income                 0
dtype: int64 

  Filled 'age' NaNs with median = 37.0
  Filled 'educational-num' NaNs with median = 10.0
  Filled 'workclass' NaNs with mode = 'Private'
  Filled 'occupation' NaNs with mode = 'Prof-specialty'
  Filled 'marital-status' NaNs with mode = 'Married-civ-spouse'
  Filled 'native-country' NaNs with mode = 'United-States'

Cleaned: 49,042 rows × 15 columns
Missing values after cleaning:
 ID                 0
age                0
workclass          0
education          0
educational-num    0
marital-status     0
occupation         0
relationship 

In [2]:
df

,ID,age,workclass,education,educational-num,marital-status,occupation,relationship,capital-gain,capital-loss,hours-per-week,income,race,sex,native-country
0,0,25.0,Private,11th,7.0,Never-married,Machine-op-inspct,Own-child,0.0,0,40,0,Black,Male,US
1,1,38.0,Private,HS-grad,9.0,Married-civ-spouse,Farming-fishing,Husband,0.0,0,50,0,White,Male,US
2,2,28.0,Local-government,Assoc-acdm,12.0,Married-civ-spouse,Protective-serv,Husband,0.0,0,40,1,White,Male,US
3,3,44.0,Private,Some-college,10.0,Married-civ-spouse,Machine-op-inspct,Husband,7688.0,0,40,1,Black,Male,US
4,4,37.0,Private,Some-college,10.0,Never-married,Prof-specialty,Own-child,0.0,0,30,0,White,Female,US
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49037,49037,57.0,Private,Some-college,10.0,Married-civ-spouse,Prof-specialty,Husband,0.0,0,40,1,White,Male,United-States
49038,49038,56.0,Self-emp-not-inc,11th,10.0,Married-civ-spouse,Other-service,Wife,0.0,0,40,0,White,Female,United-States
49039,49039,56.0,State-gov,Bachelors,13.0,Widowed,Prof-specialty,Not-in-family,0.0,0,40,0,White,Female,United-States
49040,49040,36.0,Self-emp-not-inc,Bachelors,13.0,Never-married,Craft-repair,Own-child,0.0,0,40,0,White,Male,United-States


In [6]:
df['native-country'].value_counts()

,count
native-country,
United-States,44769
Mexico,952
Philippines,298
Germany,207
Puerto-Rico,189
Canada,183
El-Salvador,156
India,151
Cuba,140


In [9]:
dataset = pd.read_csv('/content/adult_preprocessed_numeric.csv')

In [10]:
dataset

,age,educational-num,capital-gain,capital-loss,hours-per-week,income,workclass_Local-government,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native-country_Portugal,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia
0,-1.027727,-1.248191,-0.228881,-0.217067,-0.034160,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
1,-0.040203,-0.436160,-0.228881,-0.217067,0.773112,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
2,-0.799837,0.781888,-0.228881,-0.217067,-0.034160,1,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0.415578,-0.030144,2.813534,-0.217067,-0.034160,1,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
4,-0.116166,-0.030144,-0.228881,-0.217067,-0.841431,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49037,1.403102,-0.030144,-0.228881,-0.217067,-0.034160,1,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
49038,1.327139,-0.030144,-0.228881,-0.217067,-0.034160,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
49039,1.327139,1.187904,-0.228881,-0.217067,-0.034160,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
49040,-0.192130,1.187904,-0.228881,-0.217067,-0.034160,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [13]:
from sklearn.model_selection import train_test_split
x = dataset.drop('income', axis = 1)
y = dataset.income
xtrain, xtest, ytrain, ytest = train_test_split(x, y, stratify=y, test_size=0.2, random_state = 42)

In [14]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100)
model.fit(xtrain, ytrain)

RandomForestClassifier()

In [15]:
from sklearn.metrics import accuracy_score
ypreds = model.predict(xtest)
print('accuracy is : ', accuracy_score(ypreds, ytest))

accuracy is :  0.8427974309307779


In [16]:
from xgboost import XGBClassifier
model2 = XGBClassifier()
model2.fit(xtrain, ytrain)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [17]:
accuracy_score(model2.predict(xtest), ytest)

0.8690998063003365

In [23]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# 1. Calculate the class imbalance ratio to pass to 'scale_pos_weight'
# This helps the model pay more attention to the minority class (income > 50K)
pos_weight = (ytrain == 0).sum() / (ytrain == 1).sum()

# 2. Define a base model with early stopping enabled
xgb_model = XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=pos_weight,
    eval_metric='logloss',
    early_stopping_rounds=20, # Stops training if validation score doesn't improve for 20 rounds
    random_state=42
)

# 3. Define the parameter grid to search through
param_grid = {
    'n_estimators': [100, 300, 500],        # Number of trees
    'learning_rate': [0.01, 0.05, 0.1, 0.2], # Step size shrinkage
    'max_depth': [3, 5, 7, 9],               # Maximum depth of a tree
    'subsample': [0.7, 0.8, 0.9, 1.0],       # Fraction of observations to randomly sample for each tree
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],# Fraction of columns to randomly sample for each tree
    'gamma': [0, 1, 5],                      # Minimum loss reduction required to make a split
    'reg_alpha': [0, 0.1, 1],                # L1 regularization term on weights
    'reg_lambda': [0, 1, 10]                 # L2 regularization term on weights
}

# 4. Setup RandomizedSearchCV
# Using RandomizedSearchCV instead of GridSearchCV saves time while finding near-optimal params
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=20,             # Number of parameter settings that are sampled (increase for better search, but takes longer)
    scoring='accuracy',    # We optimize for accuracy, but you can change this to 'roc_auc' or 'f1'
    cv=3,                  # 3-fold cross-validation
    verbose=1,
    n_jobs=-1,             # Use all available CPU cores
    random_state=42
)

# 5. Fit the model
# We must pass the validation set to the fit method so early stopping works
fit_params = {
    "eval_set": [(xtest, ytest)],
    "verbose": False
}

print("Starting Hyperparameter Tuning...")
random_search.fit(xtrain, ytrain, **fit_params)

# 6. Extract the best model
best_xgb_model = random_search.best_estimator_

print("\nBest Hyperparameters Found:")
print(random_search.best_params_)

# 7. Evaluate the improved model
ypreds_xgb = best_xgb_model.predict(xtest)
ypreds_proba = best_xgb_model.predict_proba(xtest)[:, 1]

print('\n--- Evaluation Metrics ---')
print('Improved Accuracy: {:.4f}'.format(accuracy_score(ytest, ypreds_xgb)))
print('ROC-AUC Score: {:.4f}'.format(roc_auc_score(ytest, ypreds_proba)))
print('\nClassification Report:')
print(classification_report(ytest, ypreds_xgb))

Starting Hyperparameter Tuning...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best Hyperparameters Found:
{'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.2, 'gamma': 0, 'colsample_bytree': 0.7}

--- Evaluation Metrics ---
Improved Accuracy: 0.8334
ROC-AUC Score: 0.9220

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.83      0.88      7462
           1       0.61      0.84      0.71      2347

    accuracy                           0.83      9809
   macro avg       0.78      0.83      0.80      9809
weighted avg       0.86      0.83      0.84      9809



In [18]:
import tensorflow as tf

In [22]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load the preprocessed numeric dataset
dataset = pd.read_csv('adult_preprocessed_numeric.csv')

# Separate features (X) and target (y)
X = dataset.drop('income', axis=1).values
y = dataset['income'].values

# Split the data (using stratify to maintain the class balance ratio in train/test)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# 2. Handle Class Imbalance
# This tells the network to pay more attention to the minority class (>50K)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

print("Class Weights applied:", class_weight_dict)

# 3. Build the Neural Network Architecture
model = Sequential([
    # Input layer & First Hidden Layer
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(), # Stabilizes and accelerates training
    Dropout(0.3),         # Randomly turns off 30% of neurons to prevent overfitting

    # Second Hidden Layer
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    # Third Hidden Layer
    Dense(32, activation='relu'),
    Dropout(0.2),

    # Output Layer (Sigmoid for binary classification: outputs a probability between 0 and 1)
    Dense(1, activation='sigmoid')
])

# 4. Compile the Model
# We track AUC in addition to accuracy, as it's a better metric for imbalanced data
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 5. Define Early Stopping
# This monitors the validation loss and stops training if it doesn't improve for 10 epochs,
# then restores the best weights so you don't end up with an overfitted model.
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# 6. Train the Model
print("\nStarting Neural Network Training...")
history = model.fit(
    X_train, y_train,
    epochs=100,           # Set high, but early stopping will cut it off when optimal
    batch_size=64,        # Number of samples processed before updating weights
    validation_split=0.2, # Use 20% of training data for validation during training
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1
)

# 7. Evaluate the Model
print("\n--- Final Model Evaluation ---")
# Get raw probabilities
y_pred_proba = model.predict(X_test)
# Convert probabilities to binary classes (0 or 1) using a 0.5 threshold
y_pred_class = (y_pred_proba > 0.5).astype("int32")

print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_class))

Class Weights applied: {0: np.float64(0.6572352330217442), 1: np.float64(2.089974430002131)}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Starting Neural Network Training...
Epoch 1/100
491/491 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.7141 - auc: 0.8309 - loss: 0.5063 - val_accuracy: 0.7785 - val_auc: 0.9000 - val_loss: 0.4033
Epoch 2/100
491/491 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7846 - auc: 0.8923 - loss: 0.4104 - val_accuracy: 0.7904 - val_auc: 0.9026 - val_loss: 0.3918
Epoch 3/100
491/491 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7958 - auc: 0.9008 - loss: 0.3947 - val_accuracy: 0.7939 - val_auc: 0.9038 - val_loss: 0.3925
Epoch 4/100
491/491 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7991 - auc: 0.9025 - loss: 0.3925 - val_accuracy: 0.7883 - val_auc: 0.9049 - val_loss: 0.3937
Epoch 5/100
491/491 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7942 - auc: 0.9028 - loss: 0.3906 - val_accuracy: 0.8062 - val_auc: 0.9049 - val_loss: 0.3745
Epoch 6/100
491/491 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8092 - auc: 0.9092 - loss: 0.3798 - val_accuracy: 0.8009 - val_auc: 0.9050 - val_loss: 0.